# ЛР-1. «Hello, agent» — минимальный цикл агента

**Модуль 1 — Основы · Неделя 1**
Теги: `⚙️ #инженерия` · `[трен.] [инд.]`

> Курс: *Инженерия агентных и мультиагентных систем* (3 курс).
> Это первая лабораторная блока «Модуль 1». Парная к ней — ЛР-2 (Gridworld / MDP).

---

## 0. О чём эта тема

Курс открывается простым, но принципиальным вопросом: **что такое агент?** Мы держим в голове два
определения и весь семестр пользуемся обоими.

* **Классическое (Wooldridge):** агент — это сущность, которая *автономна*, *реактивна* (реагирует на
  среду), *проактивна* (преследует цели) и *социальна* (взаимодействует с другими).
* **Операциональное (Anthropic, 2024):** агент — это система, где **LLM управляет потоком исполнения**:
  сама решает, какой инструмент вызвать, когда остановиться, как реагировать на результат.

Между «workflow» (жёстко заданная цепочка шагов) и «агентом» (модель сама ведёт цикл) проходит главная
граница курса. Сегодня мы строим самый маленький из возможных агентов и впервые видим **agent loop** —
цикл `Thought → Action → Observation`, описанный в статье **ReAct** (Yao et al., 2022).

**Зачем именно так:** прежде чем рассуждать об агенте как о (частично наблюдаемом) MDP — а это тема
ЛР-2 — полезно своими руками увидеть, что «агент» это не магия, а цикл из ~30 строк: модель предлагает
действие, среда возвращает наблюдение, цикл повторяется.


> **🧵 Связь с проектом.** Скелет `agent loop` и трейсинг отсюда — это **основа вашего проектного агента**. Доступ к модели берите из общего `course_llm.py` (см. [PROJECT.md](../PROJECT.md)).

## 1. Цель и результат

**Цель.** Запустить минимального агента (модель + один инструмент + одна задача) и зафиксировать
его ReAct-трейс.

**Студент узнаёт / понимает (знание):**
- чем операциональное определение агента (LLM управляет потоком) отличается от классического
  [Wooldridge] и почему оба важны;
- что такое цикл ReAct и почему «agent loop» — это не магия, а `while` с парсингом ответа модели
  [[ReAct, Yao et al. 2022]](https://arxiv.org/abs/2210.03629);
- где проходит граница между *workflow* и *агентом*
  [[Anthropic, Building Effective Agents]](https://www.anthropic.com/research/building-effective-agents).

**Студент умеет (навык):**
1. развернуть воспроизводимое окружение с доступом к LLM (облако и/или локальный fallback);
2. реализовать простейший agent loop с вызовом инструмента;
3. читать и размечать трейс агента по фазам `Thought / Action / Observation`;
4. объяснить, в какой момент цикла LLM «принимает решение».

**Прогрессивное усложнение (для разных скоростей):**
🟢 *база* — запустить baseline на `MockLLM`; 🟡 *обязательное* — задания 1–3;
🔴 *углублённое (успели раньше)* — задание 4 со ★ (провокация отказов);
🔭 *rabbit-hole* — разбор `mini-SWE-agent` и OpenAI *Practical Guide to Building Agents* (см. §10).

**Артефакты на сдачу:** заполненный ноутбук + сохранённый JSON-трейс одного прогона + краткая
рефлексия (5–7 предложений).


## 2. Требования к среде

* Python 3.10+
* Пакеты: `openai` **или** `anthropic` (по выбору провайдера); опционально `requests` для локального
  сервера (Ollama / LM Studio).
* **API-ключ** облачного провайдера в переменной окружения, **или** локальная модель (Gemma/Qwen) через
  OpenAI-совместимый endpoint, **или** встроенный `MockLLM` (работает без сети — для отладки логики цикла).

> ⚠️ Не храните ключи в коде ноутбука. Используйте переменные окружения:
> `export OPENAI_API_KEY=...` или `export ANTHROPIC_API_KEY=...` перед запуском Jupyter.


In [ ]:
# Установка (раскомментируйте нужное). В аудитории обычно уже предустановлено.
# %pip install openai            # для OpenAI-совместимых endpoint (вкл. Ollama/LM Studio)
# %pip install anthropic         # для Claude
import os, json, re, textwrap
print("Python OK. Доступные ключи:",
      "OPENAI" if os.getenv("OPENAI_API_KEY") else "-",
      "ANTHROPIC" if os.getenv("ANTHROPIC_API_KEY") else "-")


## 3. Baseline: провайдер-агностичный клиент LLM

Чтобы лаборатория запускалась у всех — с облаком, с локальной моделью или вообще без сети — введём
тонкую обёртку `chat(messages) -> str`. Реализованы три бэкенда:

* `OpenAIBackend` — любой OpenAI-совместимый API (OpenAI, Together, локальный Ollama/LM Studio).
* `AnthropicBackend` — Claude.
* `MockLLM` — **детерминированная заглушка** без сети. Она имитирует ReAct-разметку, чтобы вы могли
  отладить сам цикл и разметку трейса до подключения реальной модели.

> 💡 Начните с `MockLLM`, убедитесь, что цикл работает, затем переключитесь на реальный бэкенд одной
> строкой в ячейке «Сборка агента».


In [ ]:
class MockLLM:
    """Оффлайн-заглушка. Возвращает ReAct-разметку, эмулируя вызов калькулятора.
    Нужна, чтобы отладить agent loop без ключей и без сети."""
    name = "mock"
    def __init__(self):
        self._step = 0
    def chat(self, messages):
        # Достаём последнюю реплику пользователя/наблюдения
        last = messages[-1]["content"]
        if "Observation:" not in last and self._step == 0:
            self._step += 1
            # Пытаемся вытащить арифметическое выражение из задачи
            # вытаскиваем арифметическое выражение из текста задачи, напр. "(12 + 8) * 3 - 4"
            # (цифры, скобки и знаки + - * / ); это нужно только заглушке, реальной модели — нет
            m = re.search(r"([0-9\(][0-9\s\.\+\-\*\/\(\)]*[0-9\)])", last)
            expr = m.group(1).strip() if m else "2+2"
            return (f"Thought: нужно вычислить выражение {expr}. Вызову калькулятор.\n"
                    f"Action: calculator({expr})")
        # После observation — формулируем финальный ответ
        m = re.search(r"Observation:\s*([\-0-9\.]+)", last)
        val = m.group(1) if m else "?"
        return f"Thought: получил результат {val}. Это и есть ответ.\nFinal Answer: {val}"


class OpenAIBackend:
    """OpenAI-совместимый бэкенд. base_url можно указать на локальный Ollama/LM Studio."""
    def __init__(self, model="gpt-4o-mini", base_url=None):
        from openai import OpenAI
        self.client = OpenAI(base_url=base_url) if base_url else OpenAI()
        self.model = model
        self.name = f"openai:{model}"
    def chat(self, messages):
        r = self.client.chat.completions.create(model=self.model, messages=messages, temperature=0)
        return r.choices[0].message.content


class AnthropicBackend:
    def __init__(self, model="claude-3-5-haiku-latest"):
        import anthropic
        self.client = anthropic.Anthropic()
        self.model = model
        self.name = f"anthropic:{model}"
    def chat(self, messages):
        sys = "\n".join(m["content"] for m in messages if m["role"] == "system")
        conv = [m for m in messages if m["role"] != "system"]
        r = self.client.messages.create(model=self.model, system=sys, messages=conv,
                                         max_tokens=512, temperature=0)
        return r.content[0].text


## 4. Единственный инструмент: калькулятор

В курсе мы будем подключать `read_file`, `run_bash` и целые MCP-серверы (ЛР-4, ЛР-5). Сегодня —
один безопасный инструмент. Обратите внимание: инструмент — это **обычная функция Python** плюс
описание, которое попадёт в промпт. Никакой магии.


In [ ]:
def calculator(expression: str) -> str:
    """Безопасно вычисляет арифметическое выражение. Возвращает строку с результатом."""
    # Разрешаем только цифры, операторы и скобки — защита от eval-инъекций.
    if not re.fullmatch(r"[0-9\.\+\-\*\/\(\)\s]+", expression):
        return "ERROR: недопустимые символы в выражении"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"ERROR: {e}"

TOOLS = {"calculator": calculator}

# Самопроверка инструмента
assert calculator("2 + 2*3") == "8"
assert calculator("__import__('os')").startswith("ERROR")
print("Инструмент calculator работает.")


## 5. Agent loop (≈30 строк)

Вот всё ядро агента. Прочитайте его построчно — это шаблон, к которому мы будем возвращаться весь
семестр. Цикл: дать модели контекст → распарсить её ответ → если это вызов инструмента, выполнить и
вернуть `Observation`, иначе закончить.


In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""\
    Ты — агент, решающий задачу по шагам в формате ReAct.
    На каждом шаге выводи строго:
      Thought: <рассуждение>
      Action: calculator(<выражение>)
    Когда ответ готов, выведи:
      Final Answer: <ответ>
    Доступный инструмент: calculator(expression) — вычисляет арифметику.
""")

ACTION_RE = re.compile(r"Action:\s*calculator\((.*)\)", re.S)
FINAL_RE  = re.compile(r"Final Answer:\s*(.*)", re.S)

def run_agent(llm, task, max_steps=5, verbose=True):
    """Минимальный ReAct-цикл. Возвращает (answer, trace)."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": task}]
    trace = []
    for step in range(max_steps):
        reply = llm.chat(messages)
        trace.append({"step": step, "type": "llm", "content": reply})
        if verbose: print(f"--- step {step} ---\n{reply}\n")

        final = FINAL_RE.search(reply)
        if final:
            return final.group(1).strip(), trace

        act = ACTION_RE.search(reply)
        if act:
            obs = TOOLS["calculator"](act.group(1).strip())
            trace.append({"step": step, "type": "observation", "content": obs})
            if verbose: print(f"Observation: {obs}\n")
            messages.append({"role": "assistant", "content": reply})
            messages.append({"role": "user", "content": f"Observation: {obs}"})
        else:
            return "(агент не вернул ни Action, ни Final Answer)", trace
    return "(достигнут лимит шагов)", trace


## 6. Сборка и запуск

По умолчанию используем `MockLLM` — запускается без ключей. Чтобы подключить реальную модель,
раскомментируйте нужную строку.


In [ ]:
# Выберите бэкенд:
llm = MockLLM()
# llm = OpenAIBackend(model="gpt-4o-mini")
# llm = OpenAIBackend(model="qwen2.5", base_url="http://localhost:11434/v1")  # локальный Ollama
# llm = AnthropicBackend(model="claude-3-5-haiku-latest")

print("Backend:", llm.name)
task = "Сколько будет (12 + 8) * 3 - 4?"
answer, trace = run_agent(llm, task)
print("\nИТОГ:", answer)


## 7. Задания

**Задание 1 (обязательное).** Запустите агента на задаче-примере и сохраните трейс в файл.


In [ ]:
# TODO 1: сохраните трейс прогона в JSON-файл trace_run.json
with open("trace_run.json", "w", encoding="utf-8") as f:
    json.dump(trace, f, ensure_ascii=False, indent=2)
print("Трейс сохранён, шагов:", len(trace))


**Задание 2 (обязательное).** Разметьте трейс по фазам ReAct. Заполните функцию `annotate`.


In [ ]:
def annotate(trace):
    """TODO 2: верните список (phase, content), где phase ∈ {Thought, Action, Observation, Final}.
    Пример разбора строки LLM по префиксам:
        import re
        for line in reply.splitlines():
            m = re.match(r"(Thought|Action|Final Answer):\s*(.*)", line)
            if m: labeled.append((m.group(1), m.group(2)))
    Для записи type=='observation' это просто фаза 'Observation'."""
    labeled = []
    # --- ваш код ---
    return labeled

# for phase, content in annotate(trace):
#     print(f"[{phase}] {content[:80]}")


**Задание 3 (обязательное).** Подключите **реальную** модель и прогоните ту же задачу.
Пошагово: (1) задайте ключ в терминале до запуска Jupyter — `export OPENAI_API_KEY=...`
(или `ANTHROPIC_API_KEY`); (2) в ячейке «Сборка агента» раскомментируйте нужный бэкенд;
(3) сравните с `MockLLM` по двум осям: число шагов `len(trace)` и формулировки `Thought`
(у реальной модели они богаче и могут идти без явного префикса). Подсказка по сравнению — ниже.

**Задание 4 (со звёздочкой).** Спровоцируйте отказ одним из трёх способов. Конкретные идеи:
(а) **галлюцинация аргумента** — попросите «вычисли (1/3)*3 и округли до 10 знаков», смотрите, не
придумает ли агент несуществующий аргумент; (б) **игнорирование observation** — вручную подмешайте
в контекст мусорное `Observation: XYZ` и проверьте, заметит ли; (в) **зацикливание** — снизьте
`max_steps=2` на задаче, требующей больше шагов. Сохраните трейс отказа — пригодится в ЛР-4.


In [ ]:
# TODO 3: сравнение трейсов mock vs реальная модель
def compare_traces(t1, t2):
    print(f"шагов: mock={sum(1 for x in t1 if x['type']=='llm')}  "
          f"real={sum(1 for x in t2 if x['type']=='llm')}")
    # выведите рядом первые Thought из каждого трейса
# TODO 4: ваш сценарий отказа здесь


## 8. Рефлексия (обязательная, 5–7 предложений)

Ответьте текстом в ячейке ниже:
1. В какой именно момент цикла LLM «принимает решение»? Что было бы, если убрать `Observation` из
   контекста?
2. Чем ваш агент отличается от обычного workflow (жёсткой цепочки `parse → calc → print`)?


_Ваша рефлексия:_



## 9. Критерии оценки (тренировочная, не входит в 20%)

| Критерий | Вес |
|---|---|
| Окружение воспроизводимо, агент запускается (mock **или** реальная модель) | 40% |
| Трейс сохранён и корректно размечен по фазам (`annotate`) | 40% |
| Рефлексия: где в цикле принимается решение | 20% |

## 10. Референсы и доп. литература

**Обязательное чтение к неделе:**
* ★ Anthropic — *Building Effective Agents* (Schluntz & Zhang, дек. 2024). https://www.anthropic.com/research/building-effective-agents
* ★ Yao et al. (2022) — *ReAct*. https://arxiv.org/abs/2210.03629
* AIMA (Russell & Norvig, 4th ed.), гл. 2 §2.1–2.4. http://aima.cs.berkeley.edu/

**Рекомендуется / углубление:**
* ○ Wooldridge — *An Introduction to MultiAgent Systems*, гл. 2. https://www.cs.ox.ac.uk/people/michael.wooldridge/pubs/imas/teaching.html
* ○ OpenAI — *A Practical Guide to Building Agents* (2025). https://platform.openai.com/docs/guides/agents
* 🔭 `mini-SWE-agent` (~100 строк) — задел к ЛР-4. https://github.com/SWE-agent/mini-swe-agent

Полная библиография курса: [`../REFERENCES.md`](../REFERENCES.md).

**Связь с курсом:** → ЛР-2 (тот же цикл как MDP), → ЛР-4 (агент в 100 строк с реальными инструментами),
→ ЛР-5 (инструменты через MCP).


## ✅ Самопроверка (мини-автогрейдер)

Запустите после выполнения заданий. В исходном ноутбуке часть пунктов будет ❌ — это ожидаемо, пока задания не сделаны.

In [ ]:
print("="*10, "САМОПРОВЕРКА", "="*10)
print("✅ — зачтено, ❌ — ещё не готово (нормально до выполнения заданий)\n")
def _ck(name, fn):
    try: ok = bool(fn())
    except Exception: ok = False
    print(("✅" if ok else "❌"), name)
_ck("baseline: агент дал ответ", lambda: bool(answer))
_ck("Зад.2: annotate размечает фазы ReAct", lambda: len(annotate(trace))>=2 and all(p in {"Thought","Action","Observation","Final Answer"} for p,_ in annotate(trace)))
